## Structure Output

* Pydantic approach = pehle se decide karna ki AI ka answer kis structure mein aayega.

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000017B6286AB90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000017B6286AAA0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
from pydantic import BaseModel, Field
class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The release year of the movie")
    genre: str = Field(..., description="The genre of the movie")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The rating of the movie")
    

In [5]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000017B6286AB90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000017B6286AAA0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title':

In [6]:
model.invoke("provide details of the movie Inception in the structured format")

AIMessage(content='<think>\nOkay, the user wants details about the movie Inception in a structured format. Let me start by recalling the basic information about the movie. I know it\'s a 2010 film directed by Christopher Nolan. The main genre is sci-fi, but it also has action and thriller elements. The lead actor is Leonardo DiCaprio as Dom Cobb. The plot revolves around entering dreams to implant or extract ideas.\n\nI need to structure this information properly. The user probably wants the data organized in sections like Title, Year, Genre, Director, Cast, Plot, Themes, Reception, Awards, etc. Let me check each part step by step.\n\nFirst, the title is "Inception," year 2010. Director is Christopher Nolan, which I have to confirm. The main cast includes DiCaprio, but who else? There\'s Joseph Gordon-Levitt, Ellen Page, Tom Hardy, and others. I should list them accurately. The plot summary needs to be concise but cover the key points: Dom Cobb\'s ability to enter dreams, the heist asp

In [8]:
model_with_structure.invoke("Please provide details about the movie 'Inception'.")


Movie(title='Inception', year=2010, genre='Science Fiction, Action', director='Christopher Nolan', rating=8.8)

# message output alongside with parsed structured output

In [10]:
# message output alongside with parsed structured output

from pydantic import BaseModel, Field
class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The release year of the movie")
    genre: str = Field(..., description="The genre of the movie")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The rating of the movie")

model_with_structure=model.with_structured_output(Movie, include_raw = True)
response = model_with_structure.invoke("Please provide details about the movie 'Inception'.")
response
    

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for details about the movie 'Inception'. Let me check what tools I have available. There's a Movie function that requires title, year, genre, director, and rating. I need to recall the information about 'Inception'. The title is obviously 'Inception'. It was released in 2010. The genre is science fiction or maybe action. The director is Christopher Nolan. As for the rating, I think it's around 8.8 on IMDb. Let me make sure those details are correct. Once I have that, I can structure the tool call with those parameters. I need to ensure all required fields are included. Let me double-check the year and director to avoid mistakes. Yeah, 2010 and Christopher Nolan are correct. The genre might be a mix of sci-fi and action, but the primary genre is sci-fi. The rating is 8.8. Alright, I can now format the tool call with these parameters.\n", 'tool_calls': [{'id': 'wwh6ra733', 'function': {'argume

# Nested Structure

In [11]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class Movie(BaseModel):
    title: str
    year: int
    actors: list[Actor]
    rating: float = Field(..., description="Movie rating")
model_with_structure=model.with_structured_output(Movie)
response = model_with_structure.invoke("Please provide details about the movie 'Inception'.")
response

Movie(title='Inception', year=2010, actors=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne')], rating=8.8)

# TypedDict

- provide a similar alternative using python built in typing , idea when you dont need runtime validation

In [14]:
from typing_extensions import TypedDict,Annotated

In [16]:
class MoviesDict(TypedDict):
    """ A dictionary representing a movie with its details."""
    title: Annotated[str, "The title of the movie"]
    year: Annotated[int, "The year the movie was released"]
    genre: Annotated[str, "The genre of the movie"]
    director: Annotated[str, "The director of the movie"]
    rating: Annotated[float, "The rating of the movie out of 10"]

In [19]:
model_withtyped_dict=model.with_structured_output(MoviesDict)
response = model_withtyped_dict.invoke("Please provide details about the movie 'Inception'.")
response

{'director': 'Christopher Nolan',
 'genre': 'Science Fiction',
 'rating': 8.8,
 'title': 'Inception',
 'year': 2010}

In [20]:
model.profile

{'name': 'Qwen3 32B',
 'release_date': '2024-12-23',
 'last_updated': '2024-12-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 40960,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

# Data Classes

In [21]:
import os
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [27]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    name: str = Field(..., description="The name of the contact")
    email: str = Field(..., description="The email address of the contact")
    phone: str = Field(..., description="The phone number of the contact")
agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContactInfo
)
result = agent.invoke({
    "messages": [{"role": "user", "content": "extract the info from; John Doe,john@example.com,555-1234"}]

})
result

{'messages': [HumanMessage(content='extract the info from; John Doe,john@example.com,555-1234', additional_kwargs={}, response_metadata={}, id='4b4e6240-a7d3-4290-a84f-9c7a110cacc6'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user provided a string with John Doe\'s information: "John Doe,john@example.com,555-1234". I need to extract the name, email, and phone number.\n\nFirst, I\'ll split the string by commas. The first part is the name, which is "John Doe". The second part is the email, "john@example.com". The third part is the phone number, "555-1234". \n\nI should check if all required fields are present. The function ContactInfo requires name, email, and phone. All are here. \n\nNow, I\'ll map each part to the corresponding parameter. Name is straightforward. Email looks valid. The phone number has a hyphen, but the function doesn\'t specify format, so I\'ll include it as is. \n\nNo need to validate the email or phone here since the user

In [28]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='555-1234')

In [29]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    name: str = Field(..., description="The name of the contact")
    email: str = Field(..., description="The email address of the contact")
    phone: str = Field(..., description="The phone number of the contact")
agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContactInfo
)
result = agent.invoke({
    "messages": [{"role": "user", "content": "extract the info from; John Doe,john@example.com,555-1234"}]

})
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='555-1234')

# Data Class

In [36]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo(BaseModel):
    name: str 
    email: str
    phone: str
agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContactInfo
)
result = agent.invoke({
    "messages": [{"role": "user", "content": "extract the info from; John Doe,john@example.com,555-1234"}]

})
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='555-1234')